# 12 - Dual-Head Blending Hybrid

A hybrid designed to be strong on **both** rating accuracy and ranking. It fuses the
best signals - **Content-Based (Genome)**, User-kNN, Item-kNN, SVD, **LightGCN** (+ side
features) - through **two heads** trained by blending on the validation split:

- **rating head** (`Ridge`) → predicts the rating  ⇒ RMSE / MAE
- **rank head** (`LogisticRegression` on `rating ≥ 4`) → predicts relevance  ⇒ P/R/F1@K

Each task is served by its specialised head. Additive: the base models (notebooks 04-11)
are loaded frozen; only two tiny meta-models are trained here.

> Run notebooks 04-07, 10 (genome) and 11 (LightGCN) first.

In [1]:
import sys
sys.path.insert(0, "../src")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.evaluation.report import full_metrics, save_metric, top_n
from hybrid_recsys.config import RANDOM_STATE

FIGURES_DIR = Path("../artifacts/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.write_html(str(FIGURES_DIR / f"{name}.html"))
    try:
        fig.write_image(str(FIGURES_DIR / f"{name}.png"), width=1100, height=550, scale=2)
    except Exception:
        pass
    fig.show()


EVAL_USERS, N_NEGATIVES = 1_000, 100
rng = np.random.default_rng(RANDOM_STATE)

movies           = load_processed("movies")
train, val, test = load_splits()
train_val        = pd.concat([train, val], ignore_index=True)
all_movie_ids    = movies["movieId"].values
user_ratings_map = (
    train.groupby("userId").apply(lambda d: dict(zip(d["movieId"], d["rating"]))).to_dict()
)
eval_user_ids = rng.choice(
    test["userId"].unique(), size=min(EVAL_USERS, test["userId"].nunique()), replace=False
)
test_sample = test[test["userId"].isin(eval_user_ids)]
demo_user   = max(eval_user_ids, key=lambda u: len(user_ratings_map.get(u, {})))
print(f"Loaded splits - train {len(train):,}, test {len(test):,}; demo user = {int(demo_user)}")


Loaded splits - train 19,936,012, test 2,633,022; demo user = 127311


In [2]:
def ranking_curve(metrics, title):
    rows = [{"K": k, "Metric": m.capitalize(), "Value": metrics[f"k{k}"][m]}
            for k in [5, 10, 20] for m in ["precision", "recall", "f1"]]
    return px.line(pd.DataFrame(rows), x="K", y="Value", color="Metric", markers=True,
                   title=f"Ranking metrics @ K - {title}")


def error_hist(preds, title):
    err = test["rating"].to_numpy() - preds
    err = err[~np.isnan(err)]
    fig = px.histogram(err, nbins=50, title=f"Rating error (true - predicted) - {title}")
    fig.update_layout(showlegend=False, xaxis_title="true - predicted", bargap=0.02)
    return fig


def show_example(predict_fn, n=10, n_candidates=3000):
    seen = set(user_ratings_map.get(demo_user, {}))
    cand = rng.choice(all_movie_ids, size=min(n_candidates, len(all_movie_ids)), replace=False)
    recs = top_n(predict_fn, demo_user, seen, cand, movies, n=n)
    hist = (
        pd.DataFrame({"movieId": list(seen),
                      "rating": [user_ratings_map[demo_user][m] for m in seen]})
        .merge(movies[["movieId", "clean_title", "genres"]], on="movieId", how="left")
        .sort_values("rating", ascending=False).head(n)
    )
    return hist, recs


def star_graph(center, leaves, weights, title, name):
    k = len(leaves)
    angles = np.linspace(0, 2 * np.pi, k, endpoint=False)
    lx, ly = np.cos(angles), np.sin(angles)
    edge_x, edge_y = [], []
    for x, y in zip(lx, ly):
        edge_x += [0, x, None]
        edge_y += [0, y, None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(color="#cccccc", width=1), hoverinfo="none"))
    fig.add_trace(go.Scatter(
        x=lx, y=ly, mode="markers+text",
        marker=dict(size=16, color=list(weights), colorscale="Teal",
                    showscale=True, colorbar=dict(title="sim")),
        text=[f"{l}<br>{w:.2f}" for l, w in zip(leaves, weights)],
        textposition="top center", hoverinfo="text"))
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text",
                             marker=dict(size=26, color="#EF553B"),
                             text=[center], textposition="bottom center", hoverinfo="text"))
    fig.update_layout(title=title, showlegend=False,
                      xaxis=dict(visible=False), yaxis=dict(visible=False))
    save_fig(fig, name)
    return fig


## Load the frozen base models & define the feature builder

In [3]:
from hybrid_recsys.models.content import ContentBasedRecommender
from hybrid_recsys.models.collaborative import SVDModel, ItemKNNModel, UserKNNModel
from hybrid_recsys.models.lightgcn import LightGCNRecommender
from hybrid_recsys.config import ARTIFACTS_MODELS

cb_g = ContentBasedRecommender.load(path=ARTIFACTS_MODELS / "content_genome_model.joblib")
uk   = UserKNNModel.load()
ik   = ItemKNNModel.load()
svd  = SVDModel.load()
lg   = LightGCNRecommender.load()

ip = train.groupby("movieId").size().to_dict()
uc = train.groupby("userId").size().to_dict()

def base_feats(u, i):
    return [cb_g.predict(user_ratings_map.get(u, {}), i),
            uk.predict(u, i), ik.predict(u, i), svd.predict(u, i), lg.predict(u, i),
            ip.get(i, 0), uc.get(u, 0), ip.get(i, 0)]

print("loaded 5 base models + side features")

loaded 5 base models + side features


## Fit the two heads on the validation split (blending)

In [4]:
from hybrid_recsys.models.hybrid import DualHeadHybrid

val_s = val.sample(min(80_000, len(val)), random_state=RANDOM_STATE)
X_val = np.array([base_feats(r.userId, r.movieId) for r in val_s.itertuples()], dtype=float)
dual = DualHeadHybrid().fit(X_val, val_s["rating"].to_numpy())
dual.save()
print("saved dual_head_hybrid.joblib")

saved dual_head_hybrid.joblib


## Rating head — RMSE / MAE on the full test set  *(slow: scores 5 base models per row)*

In [5]:
from hybrid_recsys.evaluation.metrics import evaluate_rating_prediction

X_test = np.array([base_feats(r.userId, r.movieId) for r in test.itertuples()], dtype=float)
preds  = dual.predict_rating(X_test)
rp     = evaluate_rating_prediction(test["rating"].to_numpy(), preds)
print(f"RMSE={rp['rmse']:.4f}  MAE={rp['mae']:.4f}")

RMSE=0.8028  MAE=0.6031


## Rank head — Precision / Recall / F1@K

In [6]:
from hybrid_recsys.evaluation.metrics import evaluate_ranking_sampled

rank_pred = lambda u, i: dual.rank_score_one(base_feats(u, i))
ranking = evaluate_ranking_sampled(test_sample, rank_pred, train_val,
                                   all_movie_ids=all_movie_ids, n_negatives=N_NEGATIVES,
                                   k_values=[5, 10, 20], random_state=RANDOM_STATE)
m = {"rmse": round(rp["rmse"], 4), "mae": round(rp["mae"], 4),
     **{f"k{k}": {kk: round(vv, 4) for kk, vv in v.items()} for k, v in ranking.items()}}
save_metric("Dual-Head Hybrid", m)
print(f"RMSE={m['rmse']}  MAE={m['mae']}  F1@10={m['k10']['f1']}")

RMSE=0.8028  MAE=0.6031  F1@10=0.4239


## Ranking metrics @ K

In [7]:
save_fig(ranking_curve(m, "Dual-Head Hybrid"), "eval_dualhead_ranking")

## Head weights — what each head relies on

In [8]:
w = pd.DataFrame({"feature": DualHeadHybrid.FEATURE_NAMES,
                  "rating_head (Ridge)": np.round(dual.rating_head.coef_, 4),
                  "rank_head (LogReg)":  np.round(dual.rank_head.coef_[0], 4)})
display(w)
fig = px.bar(w.melt(id_vars="feature", var_name="head", value_name="weight"),
             x="feature", y="weight", color="head", barmode="group",
             title="Dual-Head Hybrid - learned weights per head")
fig.update_layout(xaxis_tickangle=-30)
save_fig(fig, "eval_dualhead_weights")

,feature,rating_head (Ridge),rank_head (LogReg)
0,pred_content_genome,0.1223,0.2849
1,pred_user_knn,-0.0236,0.0164
2,pred_item_knn,0.2587,0.8889
3,pred_svd,0.6479,1.3257
4,pred_lightgcn,-0.0012,-0.0157
5,item_popularity,0.0000,0.0000
6,user_rating_count,-0.0000,-0.0002
7,item_rating_count,0.0000,0.0000


## Example — recommendations (ranked by relevance, annotated with predicted rating)

In [9]:
seen = set(user_ratings_map.get(demo_user, {}))
cand = [int(m) for m in rng.choice(all_movie_ids, size=2000, replace=False) if m not in seen]
F = np.array([base_feats(demo_user, m) for m in cand], dtype=float)
ex = (pd.DataFrame({"movieId": cand,
                    "relevance": np.round(dual.rank_score(F), 3),
                    "pred_rating": np.round(dual.predict_rating(F), 2)})
      .sort_values("relevance", ascending=False).head(10)
      .merge(movies[["movieId", "clean_title", "genres"]], on="movieId"))
display(ex[["clean_title", "genres", "relevance", "pred_rating"]])

,clean_title,genres,relevance,pred_rating
0,"Ascent, The (Voskhozhdeniye)",Drama|War,0.895,4.56
1,Rififi (Du rififi chez les hommes),Crime|Film-Noir|Thriller,0.877,4.47
2,Still Walking (Aruitemo aruitemo),Drama,0.859,4.45
3,Angel's Egg (Tenshi no tamago),Animation|Drama|Fantasy,0.857,4.44
4,"Man Who Sleeps, The (Un homme qui dort)",Drama,0.857,4.44
5,Parasite,Comedy|Drama,0.853,4.40
6,Wow! A Talking Fish!,Animation|Children|Comedy|Fantasy,0.840,4.47
7,"Draughtsman's Contract, The",Drama,0.838,4.38
8,Force Majeure,Comedy|Drama,0.835,4.37
9,Fellini's Casanova (Il Casanova di Federico Fe...,Drama|Romance,0.827,4.37


## Takeaway

The dual-head design aims to top **every** column: the Ridge head (a superset of the
Stacked Hybrid's inputs) for RMSE/MAE, the logistic head (fed by LightGCN + genome) for
ranking. Unlike pure LightGCN it scores **all** candidates (NaN base preds are imputed),
so it has no candidate-coverage caveat. "Best at all metrics" comes from using the right
head per task - one model, two objectives.